Install Transformers library



In [ ]:
pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 27.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 26.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 80.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.6 MB/s eta 0:00:00


Import Essential Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from transformers import BertTokenizer
import pandas as pd

the BERT model architecture

In [ ]:
class BERT(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers, num_heads, ff_dim, dropout, max_seq_length):
        super(BERT, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, hidden_size) # Token embeddings for BERT
        self.position_embedding = PositionalEncoding(hidden_size, max_seq_length) # Positional embeddings for BERT
        self.encoder = Encoder(hidden_size, num_layers, num_heads, ff_dim, dropout)# Encoder layers for BERT

    def forward(self, input_ids, attention_mask):
        embedded = self.token_embedding(input_ids)
        embedded = self.position_embedding(embedded)
        encoded = self.encoder(embedded, attention_mask)
        return encoded

positional encoding for BERT

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, hidden_size, max_seq_length):
        super(PositionalEncoding, self).__init__()
        self.hidden_size = hidden_size
        self.positional_embedding = nn.Embedding(max_seq_length, hidden_size)

    def forward(self, x):
        seq_length = x.size(1)
        positions = torch.arange(0, seq_length, device=x.device).unsqueeze(0)
        positions = positions.repeat(x.size(0), 1)
        position_embeddings = self.positional_embedding(positions)
        x = x + position_embeddings
        return x

encoder containing multiple self-attention layers

In [ ]:
class Encoder(nn.Module):
    def __init__(self, hidden_size, num_layers, num_heads, ff_dim, dropout):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList([EncoderLayer(hidden_size, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask):
        for layer in self.layers:
            x = layer(x, attention_mask)
            x = self.dropout(x)
        return x

the individual encoder layer with self-attention and feed-forward network

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, ff_dim, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(hidden_size, num_heads, dropout)
        self.ffn = FeedForwardNetwork(hidden_size, ff_dim, dropout)
        self.layer_norm1 = nn.LayerNorm(hidden_size)
        self.layer_norm2 = nn.LayerNorm(hidden_size)

    def forward(self, x, attention_mask):
        residual = x
        x = self.layer_norm1(x + self.self_attention(x, x, x, attention_mask))
        x = self.layer_norm2(x + self.ffn(x))
        return x

the multi-head attention mechanism

In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, dropout):
        super(MultiHeadAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_size = hidden_size // num_heads

        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, attention_mask):
        batch_size = query.size(0)

        # Linear transformation
        query = self.query(query)
        key = self.key(key)
        value = self.value(value)

        # Reshape for multi-head attention
        query = query.view(batch_size, -1, self.num_heads, self.head_size).transpose(1, 2)
        key = key.view(batch_size, -1, self.num_heads, self.head_size).transpose(1, 2)
        value = value.view(batch_size, -1, self.num_heads, self.head_size).transpose(1, 2)

        # Attention scores and scaled dot-product attention
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_size, dtype=torch.float32))
        # Ensure the attention mask is boolean
        attention_mask = attention_mask.bool()
        scores = scores.masked_fill(attention_mask.unsqueeze(1).unsqueeze(2).expand_as(scores), float("-inf"))
        scores = torch.softmax(scores, dim=-1)
        scores = self.dropout(scores)
        attended_values = torch.matmul(scores, value)

        # Reshape and concatenate
        attended_values = attended_values.transpose(1, 2).contiguous()
        attended_values = attended_values.view(batch_size, -1, self.hidden_size)

        return attended_values


the feed-forward network used within the encoder

In [ ]:

class FeedForwardNetwork(nn.Module):
    def __init__(self, hidden_size, ff_dim, dropout):
        super(FeedForwardNetwork, self).__init__()
        self.fc1 = nn.Linear(hidden_size, ff_dim)
        self.fc2 = nn.Linear(ff_dim, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


a custom dataset to handle reviews

In [ ]:

class ReviewDataset(Dataset):
    def __init__(self, csv_file, tokenizer, fraction=None):
        self.data = pd.read_csv(csv_file)
        # If a fraction is provided, sample that fraction of the dataset
        if fraction:
            self.data = self.data.sample(frac=fraction).reset_index(drop=True)

        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        review = str(self.data.loc[index, "text"])
        label = self.data.loc[index, "label"]

        encoding = self.tokenizer.encode_plus(
            review,
            add_special_tokens=True,
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze().bool()


        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'label': torch.tensor(label, dtype=torch.long)
        }


 Set the device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Define the parameters

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size
hidden_size = 384
num_layers = 6
num_heads = 6
ff_dim = 1536
dropout = 0.1
max_seq_length = 128
batch_size = 8
num_epochs = 3
learning_rate = 2e-5


Create the datasets and data loaders


In [ ]:

fraction = 0.10  # for example...
train_dataset = ReviewDataset('train_reviews.csv', tokenizer, fraction=fraction)
test_dataset = ReviewDataset('test_reviews.csv', tokenizer)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


Initialize the model, criterion, and optimizer

In [ ]:
model = BERT(vocab_size, hidden_size, num_layers, num_heads, ff_dim, dropout, max_seq_length)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


training loop for one epoch

In [ ]:
def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs[:, 0, :]  # Consider only the first token's output
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * input_ids.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss


 Train the model

In [ ]:
for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {train_loss:.4f}")


Epoch 1/3 - Training Loss: 1.4638
Epoch 2/3 - Training Loss: 1.4147
Epoch 3/3 - Training Loss: 1.4089


predict the sentiment of a sentence

In [ ]:
def predict_sentiment(sentence, model, tokenizer, max_seq_length):
    model.eval()

    inputs = tokenizer.encode_plus(
        sentence,
        add_special_tokens=True,
        truncation=True,
        padding='max_length',
        max_length=max_seq_length,
        return_tensors='pt'
    )

    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs[:, 0, :]  # Consider only the first token's output

    probabilities = nn.functional.softmax(logits, dim=1)
    predicted_label = torch.argmax(probabilities, dim=1).item()

    return predicted_label, probabilities.squeeze().tolist()



Save and load the model weights

In [ ]:
torch.save(model.state_dict(), 'bert_model.pth')
model.load_state_dict(torch.load('bert_model.pth'))
model.to(device)

BERT(
  (token_embedding): Embedding(30522, 384)
  (position_embedding): PositionalEncoding(
    (positional_embedding): Embedding(128, 384)
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attention): MultiHeadAttention(
          (query): Linear(in_features=384, out_features=384, bias=True)
          (key): Linear(in_features=384, out_features=384, bias=True)
          (value): Linear(in_features=384, out_features=384, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ffn): FeedForwardNetwork(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (layer_norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (layer_norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      )
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )


the maximum sequence length

In [ ]:
max_seq_length = 128  # Same as used during training

Predict the label for the sentence

In [ ]:
sentence = "I liked this movie"
predicted_label, probabilities = predict_sentiment(sentence, model, tokenizer, max_seq_length)

Display the results

In [ ]:
labels = ['Negative', 'Positive']
predicted_sentiment = labels[predicted_label]
print(f"Sentence: {sentence}")
print(f"Predicted Sentiment: {predicted_sentiment}")
print(f"Probability Distribution: {probabilities}")


Sentence: I liked this movie
Predicted Sentiment: Positive
Probability Distribution: [0.38746729493141174, 0.4482804834842682, 9.033267929225985e-07, 0.000517812732141465, 0.0004801205650437623, 0.0004888720577582717, 0.0005224455962888896, 0.0004464218800421804, 0.0005375170148909092, 0.00047452430590055883, 0.00047963776160031557, 0.0003340932889841497, 0.0003372816718183458, 0.0004921148647554219, 0.00040710074244998395, 0.00044135996722616255, 0.0003022675809916109, 0.0004066333931405097, 0.00047016071039251983, 0.0005087655154056847, 0.00037075852742418647, 0.0004933429881930351, 0.00040931490366347134, 0.0004539229266811162, 0.00034362298902124166, 0.00047481153160333633, 0.0003480711311567575, 0.00047460623318329453, 0.00048573495587334037, 0.00045153641258366406, 0.00046046890201978385, 2.297422724950593e-05, 0.00046616728650406003, 0.00046021738671697676, 0.0002613675897009671, 0.0004614231002051383, 0.0004885737434960902, 0.0004147893632762134, 0.00037038445589132607, 0.00041